### Structured Output

In [2]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FF126AB980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FF113BD0D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### Pydantic

In [17]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie object containing details about a movie"""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [18]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FF126AB980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FF113BD0D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': 'A movie object containing details about a movie', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, '

In [19]:
model.invoke("What are the details of the the movie Nayagan?")

AIMessage(content='<think>\nOkay, the user is asking about the movie "Nayagan." Let me start by recalling what I know about it. I think it\'s a Tamil film, right? The director might be Mani Ratnam, who\'s known for some big Indian movies. The main actor is Kamal Haasan, I believe. The title translates to "The Actor" in English. \n\nI should confirm the year it was released. I think it came out in 1987. The story revolves around a man who becomes a gangster and eventually turns into a politician. It\'s a character study, maybe exploring themes of power, identity, and societal issues. \n\nThe user might want details like the plot summary, director, cast, music composer, and maybe its impact or awards. They might be interested in why it\'s considered a classic. I should mention the critical acclaim, any awards it won, and its influence on Indian cinema. Also, the music by Ilaiyaraaja is probably important to note. \n\nWait, the user might not be familiar with the Indian film industry, so 

In [20]:
model_with_structure.invoke("What are the details of the the movie Nayagan?")

Movie(title='Nayagan', year=1987, director='Kamal Haasan', rating=9.5)

In [21]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)
model_with_structure

{
  raw: _ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FF126AB980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FF113BD0D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': 'A movie object containing details about a movie', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'st

In [22]:
response = model_with_structure.invoke("What are the details of the the movie Nayagan?")
print(response["raw"])
print(response["parsed"])

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the details of the movie "Nayagan." Let me see what I need to do here. The available tool is the Movie function, which requires the title, year, director, and rating. The user provided the title, but I need to find the other details.\n\nFirst, I should recall if I know any information about "Nayagan." I remember that it\'s a Tamil film. The director might be Mani Ratnam, but I\'m not entirely sure. The release year could be around the late 80s or early 90s. The rating is probably high since it\'s a classic. However, to be accurate, I should verify these details. Since I don\'t have real-time data access, I\'ll have to rely on my existing knowledge. \n\nWait, the user might expect precise information. If I\'m unsure about the director or year, maybe I should indicate that in the response. But the task is to call the Movie function with the parameters. So I need to structure the tool call with the availabl

### Nested Structure

In [ ]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    """An actor object containing details about an actor"""
    name:str=Field(description="The name of the actor")
    age:int=Field(description="The age of the actor")
    gender:str=Field(description="The gender of the actor")
    role:str=Field(description="The role the actor played in the movie")
    birth_year:int=Field(description="The year the actor was born")

class Movie(BaseModel):
    """A movie object containing details about a movie"""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")
    actors:list[Actor]=Field(description="A list of actors in the movie")
    genre: list[str]=Field(description="The genre of the movie")
    budget: float | None = Field(description="The budget of the movie in millions", default=None)